# Kidney Xenium Figure

This notebook reproduces main figures of mouse injured kidneys from the source analysis outputs using stGP and writes PNG/PDF pairs to `FigureReproducing/kidney`. The source `RealData_MouseKidneyXenium` directory is read only in this workflow.

In [1]:
from pathlib import Path
import os
import sys
import textwrap

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import seaborn as sns
import anndata as ad

BASE = Path(os.environ.get('STGP_REPRO_ROOT', Path.cwd().resolve()))
if not (BASE / 'FigureReproducing').exists():
    BASE = BASE.parent
FIG_REPRO_DIR = BASE / 'FigureReproducing'
KIDNEY_DIR = BASE / 'RealData_MouseKidneyXenium'
OUT_DIR = FIG_REPRO_DIR / 'kidney'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS = KIDNEY_DIR / 'Results' / 'stgp'
ROBUST = RESULTS / 'Robustness'
SUMMARY = ROBUST / 'Summary'
SOURCE = SUMMARY / 'source_data'

for pth in (KIDNEY_DIR, FIG_REPRO_DIR):
    pth_str = str(pth)
    if pth_str not in sys.path:
        sys.path.insert(0, pth_str)

from plot import (
    DPI,
    NM_W_FULL,
    NM_W_HALF,
    NM_W_SINGLE,
    draw_alpha_ci,
    ordered_stgp_alpha,
    p_to_stars,
    program_index,
    save_pair as save_figure_pair,
    setup_publication_style,
    spatial_program_values,
)

setup_publication_style('kidney')


def save_pair(fig, stem, *, dpi=DPI, bbox_inches='tight', pad_inches=0.04):
    return save_figure_pair(
        fig,
        stem,
        out_dir=OUT_DIR,
        dpi=dpi,
        bbox_inches=bbox_inches,
        pad_inches=pad_inches,
        vector_pdf=True,
        include_collections=True,
    )


def format_injury_time(days):
    days = float(days)
    if np.isclose(days, 1/6, atol=0.04) or np.isclose(days, 0.2, atol=0.06):
        return '4 h'
    if np.isclose(days, 0.5, atol=0.08):
        return '12 h'
    if np.isclose(days, 2.0, atol=0.25):
        return '2 d'
    if np.isclose(days, 14.0, atol=0.5):
        return '14 d'
    if np.isclose(days, 42.0, atol=2.0):
        return '6 weeks'
    if days < 1:
        return f'{days * 24:.0f} h'
    if days >= 21 and np.isclose(days / 7, round(days / 7), atol=0.15):
        return f'{int(round(days / 7))} weeks'
    return f'{days:g} d'


def ordered_slices(adata):
    obs = adata.obs.copy()
    obs['ident_str'] = obs['ident'].astype(str)
    age = obs.groupby('ident_str')['age'].first().astype(float)
    return age.sort_values().index.to_numpy(), age.sort_values().to_numpy(float)


def spatial_values(adata, program):
    return spatial_program_values(adata, program)

In [2]:
def plot_spatial_single(adata, program, title, stem, *, ncols=5, fg_dot_size=6.5):
    vals = spatial_values(adata, program)
    mids, ages = ordered_slices(adata)
    ids = adata.obs['ident'].astype(str).to_numpy()
    xy = np.asarray(adata.obsm['spatial'])
    vabs = float(np.nanpercentile(np.abs(vals), 99))
    vabs = vabs if np.isfinite(vabs) and vabs > 0 else 1.0
    nrows = int(np.ceil(len(mids) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.45 + 0.9, nrows * 2.45 + 0.55), squeeze=False)
    fig.subplots_adjust(left=0.015, right=0.905, top=0.80, bottom=0.05, wspace=0.03, hspace=0.16)
    axes_flat = axes.ravel()
    for ax in axes_flat:
        ax.axis('off')
    sc_ref = None
    for ax, mid, age in zip(axes_flat, mids, ages):
        mask = ids == mid
        sc_ref = ax.scatter(xy[mask, 0], xy[mask, 1], c=vals[mask], cmap='RdBu_r',
                            vmin=-vabs, vmax=vabs, s=fg_dot_size, linewidths=0, rasterized=False)
        ax.set_aspect('equal')
        ax.set_title(format_injury_time(age), fontsize=15, pad=3.0)
    cax = fig.add_axes([0.925, 0.18, 0.016, 0.58])
    cb = fig.colorbar(sc_ref, cax=cax)
    cb.set_label('stGP spatial embedding', fontsize=13, labelpad=7)
    cb.ax.tick_params(labelsize=11, length=3)
    fig.suptitle(title, fontsize=16, y=0.965)
    save_pair(fig, stem)


def plot_spatial_lr_combined(adata_l, adata_r, *, left_program='stGP3', right_program='stGP1'):
    vals_l = spatial_values(adata_l, left_program)
    vals_r = spatial_values(adata_r, right_program)
    ids_l = adata_l.obs['ident'].astype(str).to_numpy()
    ids_r = adata_r.obs['ident'].astype(str).to_numpy()
    xy_l = np.asarray(adata_l.obsm['spatial'])
    xy_r = np.asarray(adata_r.obsm['spatial'])
    mids_l, ages_l = ordered_slices(adata_l)
    mids_r, ages_r = ordered_slices(adata_r)
    ncols = max(len(mids_l), len(mids_r))
    vabs = float(np.nanpercentile(np.abs(np.r_[vals_l, vals_r]), 99))
    vabs = vabs if np.isfinite(vabs) and vabs > 0 else 1.0

    fig, axes = plt.subplots(2, ncols, figsize=(ncols * 2.35 + 0.35, 6.35), squeeze=False)
    fig.subplots_adjust(left=0.015, right=0.995, top=0.84, bottom=0.18, wspace=0.03, hspace=0.56)
    for ax in axes.ravel():
        ax.axis('off')
    sc_ref = None
    for ax, mid, age in zip(axes[0], mids_l, ages_l):
        mask = ids_l == mid
        sc_ref = ax.scatter(xy_l[mask, 0], xy_l[mask, 1], c=vals_l[mask], cmap='RdBu_r',
                            vmin=-vabs, vmax=vabs, s=6.5, linewidths=0, rasterized=False)
        ax.set_aspect('equal')
        ax.set_title(format_injury_time(age), fontsize=15, pad=3.0)
    for ax, mid, age in zip(axes[1], mids_r, ages_r):
        mask = ids_r == mid
        sc_ref = ax.scatter(xy_r[mask, 0], xy_r[mask, 1], c=vals_r[mask], cmap='RdBu_r',
                            vmin=-vabs, vmax=vabs, s=6.5, linewidths=0, rasterized=False)
        ax.set_aspect('equal')
        ax.set_title(format_injury_time(age), fontsize=15, pad=3.0)
    fig.text(0.505, 0.955, 'Left Kidney', ha='center', va='top', fontsize=18)
    fig.text(0.505, 0.535, 'Right Kidney', ha='center', va='top', fontsize=18)
    cax = fig.add_axes([0.25, 0.075, 0.50, 0.040])
    cb = fig.colorbar(sc_ref, cax=cax, orientation='horizontal')
    cb.set_label('stGP spatial activity', fontsize=13, labelpad=4)
    cb.ax.tick_params(labelsize=11, length=3, pad=2)
    save_pair(fig, 'kidney_immune_LstGP3_RstGP1_spatial_embedding_2x5')


def plot_day14_pair(adata_left, adata_right, left_program, right_program, left_slice, right_slice, left_title, right_title, stem, *, quantile=99.0, fixed_vabs=None, cbar_ticks=None):
    left_vals = spatial_values(adata_left, left_program)
    right_vals = spatial_values(adata_right, right_program)
    left_ids = adata_left.obs['ident'].astype(str).to_numpy()
    right_ids = adata_right.obs['ident'].astype(str).to_numpy()
    left_mask = left_ids == left_slice
    right_mask = right_ids == right_slice
    if fixed_vabs is None:
        vabs = float(np.nanpercentile(np.abs(np.r_[left_vals, right_vals]), quantile))
        vabs = vabs if np.isfinite(vabs) and vabs > 0 else 1.0
    else:
        vabs = float(fixed_vabs)

    fig, axes = plt.subplots(1, 2, figsize=(5.7, 3.8), constrained_layout=False)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.92, bottom=0.20, wspace=0.025)
    sc_ref = None
    for ax, adata, mask, vals, title in [
        (axes[0], adata_left, left_mask, left_vals, left_title),
        (axes[1], adata_right, right_mask, right_vals, right_title),
    ]:
        xy = np.asarray(adata.obsm['spatial'])
        sc_ref = ax.scatter(xy[mask, 0], xy[mask, 1], c=vals[mask], cmap='RdBu_r',
                            vmin=-vabs, vmax=vabs, s=6.2, linewidths=0, rasterized=False)
        ax.set_title(title, fontsize=16, pad=3)
        ax.set_aspect('equal')
        ax.axis('off')
    cax = fig.add_axes([0.17, 0.07, 0.66, 0.095])
    cb = fig.colorbar(sc_ref, cax=cax, orientation='horizontal')
    if cbar_ticks is not None:
        cb.set_ticks(cbar_ticks)
    elif fixed_vabs is not None:
        cb.set_ticks(np.linspace(-vabs, vabs, 5))
    cb.set_label('stGP spatial activity', fontsize=12, labelpad=3)
    cb.ax.tick_params(labelsize=10, pad=1.5)
    save_pair(fig, stem)

In [3]:
def plot_alpha_single(adata, title, stem):
    info = adata.uns['stgp']
    alpha = np.asarray(info['alpha'], dtype=float)
    p_sel = alpha.shape[0]
    x = np.arange(len(info['ages']))

    fig, axes = plt.subplots(1, p_sel, figsize=(4.2 * p_sel, 3.9), constrained_layout=False, squeeze=False)
    fig.subplots_adjust(left=0.07, right=0.99, top=0.78, bottom=0.25, wspace=0.30)
    for j, ax in enumerate(axes.ravel()):
        ages_ord, a, l, h, _ = ordered_stgp_alpha(info, j)
        labels = [format_injury_time(age) for age in ages_ord]
        draw_alpha_ci(
            ax,
            x,
            a,
            l,
            h,
            color='#2C7FB8',
            ci_line_lw=1.1,
            ci_line_alpha=0.60,
            line_lw=2.4,
            scatter_s=42,
            ci_label='95% posterior CI' if j == 0 else None,
            mean_label='Posterior mean' if j == 0 else None,
            zero_line_lw=0.8,
        )
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=12)
        ax.tick_params(axis='y', labelsize=12, length=3)
        ax.set_xlabel('Injury time', fontsize=13)
        ax.set_ylabel('Temporal effect' if j == 0 else '', fontsize=13)
        ax.set_title(f'stGP{j+1}', fontsize=15, pad=5)
        if j == 0:
            ax.legend(fontsize=10, loc='best')
    fig.suptitle(title, fontsize=22, y=0.965)
    save_pair(fig, stem)


def plot_alpha_combined(adata_l, adata_r):
    fig, axes = plt.subplots(1, 6, figsize=(20.0, 4.25), constrained_layout=False)
    fig.subplots_adjust(left=0.060, right=0.995, top=0.76, bottom=0.33, wspace=0.34)
    for block, (adata, title) in enumerate([(adata_l, 'Left Kidney'), (adata_r, 'Right Kidney')]):
        info = adata.uns['stgp']
        alpha = np.asarray(info['alpha'], dtype=float)
        x = np.arange(len(info['ages']))
        for j in range(alpha.shape[0]):
            ax = axes[block * 3 + j]
            ages_ord, a, l, h, _ = ordered_stgp_alpha(info, j)
            labels = [format_injury_time(age) for age in ages_ord]
            draw_alpha_ci(
                ax,
                x,
                a,
                l,
                h,
                color='#2C7FB8',
                ci_line_lw=1.1,
                ci_line_alpha=0.60,
                line_lw=2.4,
                scatter_s=42,
                ci_label='95% posterior CI' if block == 0 and j == 0 else None,
                mean_label='Posterior mean' if block == 0 and j == 0 else None,
                zero_line_lw=0.8,
            )
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=12)
            ax.tick_params(axis='y', labelsize=12, length=3)
            ax.set_xlabel('')
            ax.set_ylabel('Temporal effect' if j == 0 else '', fontsize=18, labelpad=8)
            ax.set_title(f'stGP{j+1}', fontsize=15, pad=5)
    axes[0].legend(fontsize=10, loc='best')
    fig.text(0.255, 0.965, 'Left Kidney', ha='center', va='top', fontsize=18)
    fig.text(0.745, 0.965, 'Right Kidney', ha='center', va='top', fontsize=18)
    fig.text(0.50, 0.060, 'Injury time', ha='center', va='center', fontsize=18)
    save_pair(fig, 'kidney_immune_alpha_trajectories_LR_1x6')


def cosine_similarity_matrix(a, b):
    shared = a.columns.intersection(b.columns)
    x = a[shared].to_numpy(float)
    y = b[shared].to_numpy(float)
    x_norm = np.linalg.norm(x, axis=1, keepdims=True)
    y_norm = np.linalg.norm(y, axis=1, keepdims=True).T
    return (x @ y.T) / np.clip(x_norm @ y_norm, 1e-12, None)


def plot_baseline_matching():
    src = pd.read_csv(SOURCE / 'fig2_left_right_similarity_source.csv')
    baseline_id = src.loc[src['is_baseline'].astype(bool), 'setting_id'].iloc[0]
    mapping = pd.read_csv(ROBUST / baseline_id / 'left_right_program_similarity.csv')
    w_l = pd.read_csv(ROBUST / baseline_id / 'Immune_L' / 'W.csv', index_col=0)
    w_r = pd.read_csv(ROBUST / baseline_id / 'Immune_R' / 'W.csv', index_col=0)
    cosmat = pd.DataFrame(cosine_similarity_matrix(w_l, w_r), index=w_l.index, columns=w_r.index)

    left_order = mapping['program_a'].astype(str).tolist()
    right_order = mapping['program_b'].astype(str).tolist()
    cosmat = cosmat.reindex(index=left_order, columns=right_order)

    fig, ax = plt.subplots(figsize=(4.2, 4.2), constrained_layout=False)
    fig.subplots_adjust(left=0.18, right=0.84, bottom=0.16, top=0.83)
    hm = sns.heatmap(
        cosmat,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        vmin=0,
        vmax=1,
        square=True,
        cbar_kws={'shrink': 0.78, 'pad': 0.030},
        annot_kws={'fontsize': 11.5},
        linewidths=0.5,
        linecolor='white',
        ax=ax,
    )
    ax.set_xlabel('Right program', fontsize=12)
    ax.set_ylabel('Left program', fontsize=12)
    ax.set_xticklabels([''] * len(right_order))
    ax.set_yticklabels([''] * len(left_order))
    ax.tick_params(axis='both', length=0)
    ax.set_title('Cosine similarity', fontsize=12.5, pad=10)
    cbar = hm.collections[0].colorbar
    cbar.set_label('')
    cbar.ax.tick_params(labelsize=10.5, length=3)
    save_pair(fig, 'kidney_fig2_left_right_baseline_matching')


def plot_gene_signature_heatmap():
    sig = pd.read_csv(SOURCE / 'fig5_baseline_signature_matrix.csv', index_col=0)
    cmap = LinearSegmentedColormap.from_list('wload', ['#FFFFFF', '#F6B5B8', '#B2182B'])
    fig, ax = plt.subplots(figsize=(NM_W_FULL * 1.18, 3.25), constrained_layout=False)
    fig.subplots_adjust(left=0.18, right=0.90, bottom=0.34, top=0.98)
    hm = sns.heatmap(sig, cmap=cmap, vmin=0, vmax=float(sig.to_numpy().max()), xticklabels=1,
                     cbar_kws={'label': 'Gene weight W', 'shrink': 0.78, 'pad': 0.035}, linewidths=0.25,
                     linecolor='white', ax=ax)
    ax.set_xlabel('Top loading genes', fontsize=13)
    ax.set_ylabel('Matched left/right programs', fontsize=13, labelpad=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8.8, fontstyle='italic')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10.5)
    ax.tick_params(length=0)
    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10, length=3)
    cbar.set_label('Gene weight W', fontsize=12, labelpad=6)
    save_pair(fig, 'kidney_fig5_baseline_gene_signatures', pad_inches=0.08)

In [4]:
def sig_label(q):
    return p_to_stars(q, nan_label='', nonsig_label='')


def plot_cn_fraction():
    df = pd.read_csv(RESULTS / 'Immune_L' / 'immune_stgp_top_decile_CN_enrichment.csv')
    cn_order = df[['CN', 'CN_short']].drop_duplicates().sort_values('CN_short')['CN'].tolist()
    cn_short = df[['CN', 'CN_short']].drop_duplicates().set_index('CN').loc[cn_order, 'CN_short'].tolist()
    prog_names = sorted(df['program'].unique(), key=program_index)
    frac_mat = df.pivot(index='program', columns='CN', values='frac_high_in_cn').reindex(index=prog_names, columns=cn_order)
    fdr_mat = df.pivot(index='program', columns='CN', values='mwu_padj').reindex(index=prog_names, columns=cn_order)
    vmax = max(0.20, float(np.nanpercentile(frac_mat.to_numpy(float), 95)))
    cmap = LinearSegmentedColormap.from_list('soft_fraction', ['#F7FBFF', '#D9EAF7', '#88BEDC', '#2C7FB8'])

    fig, ax = plt.subplots(figsize=(NM_W_FULL * 1.06, 2.85), constrained_layout=True)
    rows, cols = np.indices(frac_mat.shape)
    vals = frac_mat.to_numpy(float)
    mask = np.isfinite(vals)
    sizes = 95 + 500 * np.clip(vals / vmax, 0, 1)
    sc = ax.scatter(cols[mask], rows[mask], c=vals[mask], s=sizes[mask], cmap=cmap,
                    norm=Normalize(vmin=0, vmax=vmax), marker='o', edgecolors='none', linewidths=0)
    for i in range(fdr_mat.shape[0]):
        for j in range(fdr_mat.shape[1]):
            label = sig_label(float(fdr_mat.iloc[i, j]))
            if label:
                color = 'white' if vals[i, j] >= 0.68 * vmax else '#272727'
                ax.text(j, i, label, ha='center', va='center', fontsize=8.2, fontweight='bold', color=color)
    ax.set_xlim(-0.5, len(cn_order) - 0.5)
    ax.set_ylim(len(prog_names) - 0.5, -0.5)
    ax.set_xticks(np.arange(len(cn_order)))
    ax.set_xticklabels(cn_short, rotation=45, ha='right', rotation_mode='anchor', fontsize=11)
    ax.set_yticks(np.arange(len(prog_names)))
    ax.set_yticklabels(prog_names, fontsize=12)
    ax.set_xlabel('Cell neighborhood', fontsize=13, labelpad=3)
    ax.set_ylabel('Program', fontsize=13, labelpad=4)
    ax.tick_params(axis='both', pad=2, length=2.5, width=0.8)
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.018, ticks=[0.00, 0.05, 0.10, 0.15, 0.20])
    cbar.set_label('High-stGP cell fraction', fontsize=12, labelpad=4)
    cbar.ax.set_yticklabels(['0.00', '0.05', '0.10', '0.15', '0.20'])
    cbar.ax.tick_params(labelsize=11, length=3, width=0.8, pad=2)
    cbar.outline.set_linewidth(0.8)
    save_pair(fig, 'kidney_immune_L_stGP_CN_top5_high_score_fraction')


def plot_go_recurrence():
    recurrence = pd.read_csv(SOURCE / 'fig4_go_recurrence_matrix.csv', index_col=0)
    enrich = pd.read_csv(SOURCE / 'fig4_go_enrichment_summary.csv')
    enrich = enrich.set_index('Term_clean').reindex(recurrence.index).reset_index()

    theme_palette = {
        'chemokine_leukocyte_recruitment': '#9FB8CF',
        'immune_inflammatory_activation': '#B8A7C9',
        'injury_repair_fibrotic_remodeling': '#C9B49B',
        'cell_cycle_cytokinesis': '#A9C8A6',
        'autophagy_phagolysosome': '#D2B7A3',
        'mitochondrial_er_stress_quality_control': '#C9C3A5',
        'vesicle_trafficking_transport': '#B7C7B5',
    }
    cmap = LinearSegmentedColormap.from_list('muted_recovery_blue', ['#FAFAFA', '#E8EAED', '#CBD3E3', '#A8B8D4', '#6F86B5'])
    n = len(recurrence)
    fig_h = max(5.1, 0.34 * n + 2.2)
    fig = plt.figure(figsize=(NM_W_FULL * 1.18, fig_h), constrained_layout=False)
    gs = fig.add_gridspec(1, 3, width_ratios=[2.8, 0.85, 1.35], left=0.035, right=0.975, bottom=0.27, top=0.76, wspace=0.08)
    term_ax = fig.add_subplot(gs[0, 0])
    heat_ax = fig.add_subplot(gs[0, 1])
    enrich_ax = fig.add_subplot(gs[0, 2])

    term_ax.set_xlim(0, 1)
    term_ax.set_ylim(n, 0)
    term_ax.axis('off')
    for y, row in enrich.iterrows():
        bg = '#FAFAFA' if y % 2 == 0 else '#FFFFFF'
        theme_color = theme_palette.get(row.get('candidate_theme'), '#B8B8B8')
        term_ax.add_patch(Rectangle((0, y), 1, 1, facecolor=bg, edgecolor='white', linewidth=0.45))
        term_ax.add_patch(Rectangle((0.010, y + 0.18), 0.014, 0.64, facecolor=theme_color, edgecolor='none'))
        term_ax.text(0.040, y + 0.5, textwrap.fill(str(row['Term_clean']), width=35),
                     ha='left', va='center', fontsize=9.0, color='#2F2F2F', linespacing=0.90)
    term_ax.text(0.040, -0.48, 'GO enrichment term', ha='left', va='bottom', fontsize=11.5, color='#2F2F2F')

    heat_data = recurrence.to_numpy(float)
    im = heat_ax.pcolormesh(np.arange(heat_data.shape[1] + 1), np.arange(heat_data.shape[0] + 1),
                            heat_data, cmap=cmap, vmin=0, vmax=1, shading='flat')
    heat_ax.set_xlim(0, heat_data.shape[1])
    heat_ax.set_ylim(heat_data.shape[0], 0)
    heat_ax.set_xticks(np.arange(recurrence.shape[1]) + 0.5)
    heat_ax.set_xticklabels(['Left\nstGP3', 'Right\nstGP1'], fontsize=11.5)
    heat_ax.set_yticks([])
    heat_ax.tick_params(length=0)
    heat_ax.set_title('Recovered\nprograms', fontsize=11.5, pad=7)
    for spine in heat_ax.spines.values():
        spine.set_visible(False)
    cbar = fig.colorbar(im, ax=heat_ax, fraction=0.050, pad=0.030, ticks=[0, 0.5, 1.0])
    cbar.set_label('Recurrence fraction\nacross settings', fontsize=10.5, labelpad=5)
    cbar.ax.tick_params(labelsize=10.5, length=3)

    x_vals = enrich['max_neglog_fdr'].to_numpy(float)
    score = enrich['median_score'].fillna(0).to_numpy(float)
    s_min, s_max = float(np.nanmin(score)), float(np.nanmax(score))
    span = max(s_max - s_min, 1e-9)
    sizes = 48 + 120 * (score - s_min) / span
    y_pos = np.arange(n)
    xmin = 1.5
    xmax = max(xmin + 0.35, float(np.nanmax(x_vals)) * 1.05)
    for y, x in zip(y_pos, x_vals):
        enrich_ax.plot([xmin, max(xmin, x)], [y, y], color='#D9DDE4', lw=0.65, zorder=1)
    enrich_ax.scatter(x_vals, y_pos, s=sizes, color='#6F86B5', alpha=0.88,
                      edgecolor='white', linewidth=0.55, zorder=3)
    enrich_ax.set_xlim(xmin, xmax)
    enrich_ax.set_ylim(n - 0.5, -0.5)
    enrich_ax.set_yticks([])
    enrich_ax.xaxis.tick_top()
    enrich_ax.xaxis.set_label_position('top')
    enrich_ax.set_xlabel('GO enrichment\n-log10(FDR)', fontsize=11.5, labelpad=8)
    enrich_ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
    enrich_ax.tick_params(axis='x', labelsize=10.5, width=0.6, length=2.5, pad=1)
    enrich_ax.grid(axis='x', color='#E8E8E8', lw=0.55)
    for spine in ['left', 'right', 'bottom']:
        enrich_ax.spines[spine].set_visible(False)
    enrich_ax.spines['top'].set(linewidth=0.6, color='#777777')

    legend_scores = np.unique(np.array([
        np.floor(s_min / 500) * 500,
        np.round(np.nanmedian(score) / 500) * 500,
        np.ceil(s_max / 500) * 500,
    ]).astype(int))
    score_handles = [enrich_ax.scatter([], [], s=48 + 120 * (np.clip(s, s_min, s_max) - s_min) / span,
                                       color='#6F86B5', alpha=0.88, edgecolor='white', linewidth=0.55,
                                       label=f'{s:.0f}') for s in legend_scores]
    enrich_ax.legend(handles=score_handles, title='Score', loc='lower right', bbox_to_anchor=(1.02, -0.20),
                     fontsize=10, title_fontsize=12, frameon=False, borderpad=0.1,
                     handletextpad=0.55, labelspacing=0.55)

    used = enrich[['candidate_theme', 'candidate_theme_label']].dropna().drop_duplicates().sort_values('candidate_theme_label')
    if not used.empty:
        theme_handles = [Patch(facecolor=theme_palette.get(row.candidate_theme, '#B8B8B8'), edgecolor='none',
                               label=row.candidate_theme_label) for row in used.itertuples(index=False)]
        fig.legend(handles=theme_handles, loc='lower center', bbox_to_anchor=(0.50, 0.055),
                   ncol=min(4, len(theme_handles)), fontsize=10, frameon=False,
                   handlelength=1.0, handletextpad=0.45, columnspacing=1.0)
    fig.suptitle('GO term recurrence across robustness settings', fontsize=16, y=0.975)
    save_pair(fig, 'kidney_fig4_go_recurrence')

In [5]:
def plot_go_recurrence_original_layout():
    recurrence = pd.read_csv(SOURCE / 'fig4_go_recurrence_matrix.csv', index_col=0)
    counts = pd.read_csv(SOURCE / 'fig4_go_recurrence_counts.csv', index_col=0)
    enrich = pd.read_csv(SOURCE / 'fig4_go_enrichment_summary.csv')
    enrich = enrich.set_index('Term_clean').reindex(recurrence.index).reset_index()

    target_labels = recurrence.columns.tolist()
    theme_palette = {
        'chemokine_leukocyte_recruitment': '#9FB8CF',
        'immune_inflammatory_activation': '#B8A7C9',
        'injury_repair_fibrotic_remodeling': '#C9B49B',
        'cell_cycle_cytokinesis': '#A9C8A6',
        'autophagy_phagolysosome': '#D2B7A3',
        'mitochondrial_er_stress_quality_control': '#C9C3A5',
        'vesicle_trafficking_transport': '#B7C7B5',
        'calcium_transporter': '#B8B8B8',
    }
    cmap = LinearSegmentedColormap.from_list('muted_recovery_blue', ['#FAFAFA', '#E8EAED', '#CBD3E3', '#A8B8D4', '#6F86B5'])
    n_terms = len(recurrence)
    split_at = int(np.ceil(n_terms / 2))
    term_chunks = [recurrence.index.tolist()[:split_at], recurrence.index.tolist()[split_at:]]

    fig_w = NM_W_FULL * 1.18
    fig_h = fig_w / 1.95
    fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=False)
    gs = fig.add_gridspec(2, 6, width_ratios=[1.52, 0.72, 1.12, 1.52, 0.72, 1.12],
                          left=0.035, right=0.985, top=0.90, bottom=0.28, wspace=0.18, hspace=0.02)

    score = enrich['median_score'].fillna(0).to_numpy(float)
    score_min = float(np.nanmin(score)) if len(score) else 0.0
    score_max = float(np.nanmax(score)) if len(score) else 1.0
    score_span = max(score_max - score_min, 1e-9)
    score_to_size = lambda s: 20 + 86 * (float(s) - score_min) / score_span
    xmin = 1.5
    xmax = max(xmin + 0.35, float(enrich['max_neglog_fdr'].max()) * 1.04)
    fs = {'term': 7.2 if split_at <= 10 else 6.8, 'ratio': 7.1, 'header': 7.4, 'axis': 7.0, 'tick': 6.4}
    header_labels = ['Left\nstGP3', 'Right\nstGP1']
    summary_by_term = enrich.set_index('Term_clean')
    enrich_axes = []
    cax_ref = None

    def draw_block(terms, block_idx, show_cbar=False):
        nonlocal cax_ref
        col0 = 0 if block_idx == 0 else 3
        term_ax = fig.add_subplot(gs[:, col0])
        heat_ax = fig.add_subplot(gs[:, col0 + 1])
        enrich_ax = fig.add_subplot(gs[:, col0 + 2])
        enrich_axes.append(enrich_ax)

        block_matrix = recurrence.reindex(index=terms, columns=target_labels).fillna(0)
        block_counts = counts.reindex(index=terms, columns=target_labels).fillna(0).astype(int)
        block_summary = summary_by_term.reindex(terms).reset_index()
        n_block = len(terms)

        sns.heatmap(block_matrix, ax=heat_ax, cmap=cmap, vmin=0, vmax=1, cbar=False,
                    annot=False, linewidths=0.45, linecolor='white', xticklabels=False, yticklabels=False)
        for i in range(n_block):
            for j, label in enumerate(target_labels):
                heat_ax.text(j + 0.5, i + 0.5, f'{int(block_counts.iloc[i, j])}/12',
                             ha='center', va='center', color='white', fontsize=fs['ratio'])
        for x, label in enumerate(header_labels):
            heat_ax.text(x + 0.5, -0.52, label, ha='center', va='bottom', fontsize=fs['header'],
                         color='#2F2F2F', linespacing=0.85, clip_on=False)
        heat_ax.tick_params(axis='both', length=0, pad=0, labeltop=False, labelbottom=False)
        heat_ax.set(xlabel='', ylabel='')

        term_ax.set(xlim=(0, 1), ylim=(n_block, 0))
        term_ax.axis('off')
        for y, row in block_summary.iterrows():
            bg = '#FAFAFA' if y % 2 == 0 else '#FFFFFF'
            theme_color = theme_palette.get(row.get('candidate_theme'), '#B8B8B8')
            term_ax.add_patch(Rectangle((0, y), 1, 1, facecolor=bg, edgecolor='white', linewidth=0.40))
            term_ax.add_patch(Rectangle((0.010, y + 0.18), 0.014, 0.64, facecolor=theme_color, edgecolor='none'))
            enrich_ax.axhspan(y, y + 1, color=bg, zorder=0)
            term_ax.text(0.040, y + 0.5, textwrap.fill(str(row['Term_clean']), width=27),
                         ha='left', va='center', fontsize=fs['term'], color='#2F2F2F', linespacing=0.88)
        term_ax.text(0.040, -0.52, 'GO enrichment term', ha='left', va='bottom', fontsize=fs['header'], color='#2F2F2F')

        y_pos = np.arange(n_block) + 0.5
        x_vals = block_summary['max_neglog_fdr'].to_numpy(float)
        sizes = [score_to_size(s) for s in block_summary['median_score'].fillna(0)]
        for y, x in zip(y_pos, x_vals):
            enrich_ax.plot([xmin, max(xmin, x)], [y, y], color='#D9DDE4', lw=0.50, zorder=1)
        enrich_ax.scatter(x_vals, y_pos, s=sizes, color='#6F86B5', alpha=0.86,
                          edgecolor='white', linewidth=0.42, zorder=3)
        enrich_ax.set(xlim=(xmin, xmax), ylim=(n_block, 0), yticks=[])
        enrich_ax.xaxis.tick_top()
        enrich_ax.xaxis.set_label_position('top')
        enrich_ax.set_xlabel('GO enrichment\n-log10(FDR)', fontsize=fs['axis'], labelpad=6, color='#2F2F2F')
        enrich_ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
        enrich_ax.tick_params(axis='x', labelsize=fs['tick'], width=0.50, length=2, pad=1, colors='#4A4A4A')
        enrich_ax.grid(axis='x', color='#E8E8E8', lw=0.45)
        for spine in ['left', 'right', 'bottom']:
            enrich_ax.spines[spine].set_visible(False)
        enrich_ax.spines['top'].set(linewidth=0.50, color='#777777')
        if show_cbar:
            cax_ref = heat_ax

    for block_idx, terms in enumerate(term_chunks):
        if terms:
            draw_block(terms, block_idx, show_cbar=(block_idx == 0))

    sm = plt.cm.ScalarMappable(norm=Normalize(0, 1), cmap=cmap)
    cbar_ax = fig.add_axes([0.255, 0.190, 0.50, 0.020])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal', ticks=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
    cbar.set_label('Recovery fraction', fontsize=8.8, labelpad=3)
    cbar.ax.tick_params(labelsize=8.4, width=0.50, length=2, pad=1, colors='#4A4A4A')
    cbar.outline.set_visible(False)

    if enrich_axes:
        legend_scores = np.unique(np.array([
            np.floor(score_min / 500) * 500,
            np.round(np.nanmedian(score) / 500) * 500,
            np.ceil(score_max / 500) * 500,
        ]).astype(int))
        score_handles = [enrich_axes[-1].scatter([], [], s=score_to_size(np.clip(s, score_min, score_max)), color='#6F86B5',
                                                 alpha=0.86, edgecolor='white', linewidth=0.42, label=f'{s:.0f}')
                         for s in legend_scores]
        enrich_axes[-1].legend(handles=score_handles, title='Score', loc='lower right', bbox_to_anchor=(1.18, -0.45),
                               fontsize=9.5, title_fontsize=10.5, frameon=False, borderpad=0.1,
                               handletextpad=0.55, labelspacing=0.45)

    used = enrich[['candidate_theme', 'candidate_theme_label']].dropna().drop_duplicates().sort_values('candidate_theme_label')
    if not used.empty:
        theme_handles = [Patch(facecolor=theme_palette.get(row.candidate_theme, '#B8B8B8'), edgecolor='none',
                               label=row.candidate_theme_label) for row in used.itertuples(index=False)]
        fig.legend(handles=theme_handles, loc='lower center', bbox_to_anchor=(0.50, 0.020),
                   ncol=min(4, len(theme_handles)), fontsize=9.5, frameon=False,
                   handlelength=0.95, handletextpad=0.35, columnspacing=0.85)

    save_pair(fig, 'kidney_fig4_go_recurrence', pad_inches=0.02)

In [6]:
SLINGSHOT_GRIDNUM = 10
SLINGSHOT_MIN_CELLS = 20
SLINGSHOT_ARROW_LEN = 0.75


def slingshot_gradient_arrows(xy, pt, *, gridnum=SLINGSHOT_GRIDNUM,
                              min_cells_per_grid=SLINGSHOT_MIN_CELLS,
                              arrow_length_frac=SLINGSHOT_ARROW_LEN):
    """Grid-averaged pseudotime gradient arrows (pseudotime_Inj_PT_*.ipynb)."""
    xy = np.asarray(xy, dtype=float)
    pt = np.asarray(pt, dtype=float)
    valid = np.isfinite(pt) & np.all(np.isfinite(xy), axis=1)
    xy_valid = xy[valid]
    pt_valid = pt[valid]
    if xy_valid.size == 0:
        raise ValueError('No finite pseudotime values available for arrow overlay.')

    x_edges = np.linspace(xy_valid[:, 0].min(), xy_valid[:, 0].max(), gridnum + 1)
    y_edges = np.linspace(xy_valid[:, 1].min(), xy_valid[:, 1].max(), gridnum + 1)
    x_bin = np.clip(np.digitize(xy_valid[:, 0], x_edges) - 1, 0, gridnum - 1)
    y_bin = np.clip(np.digitize(xy_valid[:, 1], y_edges) - 1, 0, gridnum - 1)

    mean_pt = np.full((gridnum, gridnum), np.nan)
    mean_xy = np.full((gridnum, gridnum, 2), np.nan)

    for i in range(gridnum):
        for j in range(gridnum):
            mask = (x_bin == i) & (y_bin == j)
            if int(mask.sum()) >= min_cells_per_grid:
                mean_pt[i, j] = float(np.nanmean(pt_valid[mask]))
                mean_xy[i, j] = np.nanmean(xy_valid[mask], axis=0)

    arrow_start = []
    arrow_vec = []
    for i in range(gridnum):
        for j in range(gridnum):
            if not np.isfinite(mean_pt[i, j]):
                continue
            grad = np.zeros(2, dtype=float)
            center = mean_xy[i, j]
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0:
                        continue
                    ni, nj = i + di, j + dj
                    if ni < 0 or ni >= gridnum or nj < 0 or nj >= gridnum:
                        continue
                    if not np.isfinite(mean_pt[ni, nj]):
                        continue
                    direction = mean_xy[ni, nj] - center
                    dist = np.linalg.norm(direction)
                    if dist == 0:
                        continue
                    grad += (mean_pt[ni, nj] - mean_pt[i, j]) * direction / dist
            norm = np.linalg.norm(grad)
            if norm > 0:
                arrow_start.append(center)
                arrow_vec.append(grad / norm)

    arrow_start = np.asarray(arrow_start)
    arrow_vec = np.asarray(arrow_vec)
    cell_size = min(np.diff(x_edges).mean(), np.diff(y_edges).mean())
    arrow_vec = arrow_vec * cell_size * arrow_length_frac
    return arrow_start, arrow_vec


def plot_slingshot_arrow_panel(ax, adata, *, dot_size=3.0):
    xy = np.asarray(adata.obsm['spatial'])
    pt = adata.obs['slingPseudotime_1'].to_numpy(dtype=float)
    clusters = adata.obs['clusterlabel']
    if hasattr(clusters, 'cat'):
        cluster_labels = clusters.cat.categories.astype(str)
    else:
        cluster_labels = pd.Series(clusters).astype(str).unique()

    for label in cluster_labels:
        mask = clusters.astype(str).to_numpy() == str(label)
        ax.scatter(xy[mask, 0], xy[mask, 1], s=dot_size, linewidths=0, rasterized=False)

    arrow_start, arrow_vec = slingshot_gradient_arrows(xy, pt)
    if len(arrow_start) > 0:
        ax.quiver(
            arrow_start[:, 0], arrow_start[:, 1],
            arrow_vec[:, 0], arrow_vec[:, 1],
            angles='xy', scale_units='xy', scale=1,
            color='black', width=0.006,
            headwidth=4.5, headlength=6.0, headaxislength=5.0,
        )
    ax.set_aspect('equal')
    ax.axis('off')


def plot_day2_slingshot_combined(*, stem='kidney_Day2L_Day2R_slingshot_arrow_overlay_combined'):
    adata_l = ad.read_h5ad(RESULTS / 'Inj_PT_L' / 'pseudotime_Day2L' / 'Day2L_stgp_cluster_slingshot.h5ad')
    adata_r = ad.read_h5ad(RESULTS / 'Inj_PT_R' / 'pseudotime_Day2R' / 'Day2R_stgp_cluster_slingshot.h5ad')

    fig, axes = plt.subplots(1, 2, figsize=(14.0, 7.0), constrained_layout=False)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01, wspace=0.02)
    plot_slingshot_arrow_panel(axes[0], adata_l)
    plot_slingshot_arrow_panel(axes[1], adata_r)
    save_pair(fig, stem, pad_inches=0.02)

In [ ]:
adata_immune_l = ad.read_h5ad(RESULTS / 'Immune_L' / 'adata_with_scores.h5ad')
adata_immune_r = ad.read_h5ad(RESULTS / 'Immune_R' / 'adata_with_scores.h5ad')
adata_fib_l = ad.read_h5ad(RESULTS / 'Fib_L' / 'adata_with_scores.h5ad')
adata_fib_r = ad.read_h5ad(RESULTS / 'Fib_R' / 'adata_with_scores.h5ad')

# 1. Immune spatial embedding combined 2 x 5 layout.
plot_spatial_lr_combined(adata_immune_l, adata_immune_r, left_program='stGP3', right_program='stGP1')

# 3. Baseline left/right matching heatmap.
plot_baseline_matching()

# 4 and 9. Day 14 left/right pair maps with larger subplot titles.
plot_day14_pair(adata_immune_l, adata_immune_r, 'stGP2', 'stGP3', 'Day14L', 'Day14R',
                'L-stGP2', 'R-stGP3', 'kidney_immune_R_Day14_L-stGP2_R-stGP3', quantile=99.0, fixed_vabs=3)
plot_day14_pair(adata_fib_l, adata_fib_r, 'stGP2', 'stGP2', 'Day14L', 'Day14R',
                'L-stGP2', 'R-stGP2', 'kidney_fib_R_Day14_L-stGP2_R-stGP2', quantile=99.9,
                fixed_vabs=8, cbar_ticks=[-6, -3, 0, 3, 6])

# 5. Alpha trajectory combined 1 x 6 layout.
plot_alpha_combined(adata_immune_l, adata_immune_r)

# 7. Baseline gene-signature heatmap with italic gene labels.
plot_gene_signature_heatmap()

# 8. CN high-score fraction dot heatmap with sparse colorbar ticks.
plot_cn_fraction()

# 10. Keep the original two-block GO recurrence layout, with larger legends.
plot_go_recurrence_original_layout()

# 11. Day 2 slingshot arrow overlay from saved pseudotime AnnData objects.
plot_day2_slingshot_combined()

print(f'All kidney figure revisions saved to: {OUT_DIR}')